# 02 — Ray Data on the cluster (CodeFlare RayJob)

Submit the same `scale_data.py` script from Topic 1 to a **KubeRay RayJob** using the **CodeFlare SDK**.

OpenShift AI creates the Ray cluster, runs the job, and tears it down when finished.

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "extras/scripts/scale_data.py").exists():
    raise FileNotFoundError("Clone ray-workshop into this workbench and open the notebook from the repo root")

sys.path.insert(0, str(REPO_ROOT / "extras/python"))
from workshop_common import (
    LOCAL_QUEUE,
    NAMESPACE,
    RayJob,
    cpu_cluster_config,
    login,
    repo_root,
    runtime_env_for_script,
    submit_rayjob,
    wait_for_rayjob,
)

In [ ]:
# --- edit these (or set OPENSHIFT_TOKEN / OPENSHIFT_SERVER env vars) ---
OPENSHIFT_SERVER = "https://api.YOUR_CLUSTER:6443"
OPENSHIFT_TOKEN = "REPLACE_WITH_YOUR_TOKEN".strip()

auth = login(OPENSHIFT_TOKEN, OPENSHIFT_SERVER)
print(f"Namespace: {NAMESPACE}")
print(f"Local queue: {LOCAL_QUEUE}")
print(f"Repo root: {repo_root()}")

In [ ]:
cluster_config = cpu_cluster_config(num_workers=2)

job = RayJob(
    job_name="ray-workshop-scale-data",
    entrypoint="python extras/scripts/scale_data.py",
    cluster_config=cluster_config,
    namespace=NAMESPACE,
    local_queue=LOCAL_QUEUE,
    runtime_env=runtime_env_for_script(
        pip=["pyarrow>=14", "pandas>=2"],
        env_vars={
            "INPUT_PATH": "extras/data/iris.csv",
            "OUTPUT_DIR": "/tmp/ray-workshop-output/iris",
        },
    ),
    ttl_seconds_after_finished=600,
)

job_name = submit_rayjob(job)

In [ ]:
# Poll until the RayJob finishes (requires kubernetes Python package on workbench)
wait_for_rayjob(job_name)

## Verify in OpenShift AI

1. Open **OpenShift AI → Distributed workloads** (or search **RayJobs** in the console).
2. Confirm `ray-workshop-scale-data` succeeded.
3. Optional CLI from Web Terminal:

```sh
oc get rayjob -n ray-workshop
oc logs -n ray-workshop -l ray.io/node-type=head -c ray-head --tail=80
```

Look for `Done. Wrote N parquet file(s)` in the logs.